# Neighborhood DNA from Overture Maps Places
**FOSS4G Hiroshima 2026 · lightning talk companion notebook**

Replace the city name and bbox below → *Run All* → done.
No download, no ETL: DuckDB reads Overture Parquet straight off S3,
and the bbox filter runs on the storage side.

The promise from the talk, kept honest — two checks before you trust any POI metric:
1. **Control for cell size** — or you measure your data, not your city.
2. **Check the sources column** — or you compare providers, not places.

In [ ]:
# ---- the only cell you edit -------------------------------------
CITY = "hiroshima"
YMIN, XMIN, YMAX, XMAX = 34.30, 132.35, 34.48, 132.55   # lat/lon bbox

# a few bboxes to try:
#   belgrade   44.72, 20.35, 44.87, 20.55
#   amsterdam  52.30, 4.80, 52.42, 5.00
#   london     51.44, -0.22, 51.56, -0.02

In [ ]:
# %pip install duckdb==1.5.5 geopandas==1.1.1 shapely matplotlib pyarrow
import duckdb

RELEASE = "2026-07-22.0"  # `categories` is removed from Sept 2026 — use `taxonomy`
H3_RES  = 8               # equal-area cells; a degree grid stretches 1.5x with latitude
LEVEL   = 1               # taxonomy level: 1 = 13 branches, 3 = ~200 leaves
MIN_POI = 10              # below this, diversity metrics are noise, not signal

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs; INSTALL h3 FROM community; LOAD h3;")
con.sql("SET s3_region='us-west-2';")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TABLE cells AS
WITH places AS (
    SELECT h3_latlng_to_cell(bbox.ymin, bbox.xmin, {H3_RES}) AS cell,
           taxonomy.hierarchy[least({LEVEL}, len(taxonomy.hierarchy))] AS category
    FROM read_parquet(
        's3://overturemaps-us-west-2/release/{RELEASE}/theme=places/type=place/*',
        hive_partitioning = 1)
    WHERE bbox.xmin BETWEEN {XMIN} AND {XMAX}
      AND bbox.ymin BETWEEN {YMIN} AND {YMAX}
      AND taxonomy IS NOT NULL AND len(taxonomy.hierarchy) > 0
),
shares AS (
    SELECT cell, category, n, n / sum(n) OVER (PARTITION BY cell) AS p
    FROM (SELECT cell, category, count(*)::DOUBLE AS n
          FROM places GROUP BY 1, 2)
)
SELECT cell,
       h3_cell_to_boundary_wkt(cell)              AS geometry,
       sum(n)                                     AS poi_count,
       count(*)                                   AS cat_richness,
       -sum(p * ln(p))                            AS cat_shannon,
       -sum(p * ln(p)) / nullif(ln(count(*)), 0)  AS cat_evenness,
       1 - sum(p * p)                             AS cat_gini_simpson,
       exp(-sum(p * ln(p)))                       AS cat_hill_q1,
       1 / sum(p * p)                             AS cat_hill_q2,
       max(p)                                     AS top_cat_share,
       arg_max(category, p)                       AS dominant_category
FROM shares GROUP BY cell HAVING sum(n) >= {MIN_POI};
""")
con.sql(f"COPY cells TO '{CITY}_dna.parquet' (FORMAT parquet)")
con.sql("""SELECT count(*) AS cells, round(sum(poi_count)) AS poi,
    round(median(cat_hill_q1),2) AS med_hill_q1,
    round(median(top_cat_share),2) AS med_top_share,
    mode(dominant_category) AS most_common_dominant FROM cells""").df()

## Map — `top_cat_share`, "how mono-functional is the cell"
White-outlined cells: one function owns half the places. That is the shortlist.

In [ ]:
import geopandas as gpd, matplotlib.pyplot as plt
from shapely import wkt

df = con.sql("SELECT * FROM cells").df()
g = gpd.GeoDataFrame(df, geometry=df.geometry.apply(wkt.loads), crs=4326)
ax = g.plot(column="top_cat_share", cmap="viridis", vmin=0.2, vmax=0.6,
            figsize=(9, 8), legend=True,
            legend_kwds={"label": "top_cat_share — how mono-functional"})
flag = g[g.top_cat_share >= 0.5]
flag.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.2)
ax.set_title(f"{CITY}: {len(flag)} flagged of {len(g)} cells")
ax.set_axis_off()

## Check 1 — control for cell size
Richness climbs with POI count (you measure your data);
Hill q2 flattens (you measure the city). Compare cities only inside matched density bins.

In [ ]:
import pandas as pd
df["bin"] = pd.cut(df.poi_count, [10,20,40,80,160,320,10**9],
                   labels=["10-19","20-39","40-79","80-159","160-319","320+"])
df.groupby("bin", observed=True)[["cat_richness","cat_hill_q2"]].median().round(2)

## Check 2 — the sources column
Overture blends providers; their mix differs by city (Belgrade ~93% one provider).
Rankings can flip when you filter by source — check before comparing cities.

In [ ]:
con.sql(f"""
SELECT unnest(sources).dataset AS provider, count(*) AS n
FROM read_parquet(
    's3://overturemaps-us-west-2/release/{RELEASE}/theme=places/type=place/*',
    hive_partitioning = 1)
WHERE bbox.xmin BETWEEN {XMIN} AND {XMAX}
  AND bbox.ymin BETWEEN {YMIN} AND {YMAX}
GROUP BY 1 ORDER BY n DESC
""").df()

---
Slides & figures CC BY 4.0 · code MIT · Marija Ercegovac, FOSS4G Hiroshima 2026
Full caveat list: [`CAVEATS.md`](../CAVEATS.md)